# EcoHome Energy Database Setup with SQLAlchemy

Complete database initialization for the EcoHome smart energy optimization system using SQLAlchemy ORM.

This notebook will:
- Define ORM models for EnergyUsage and SolarGeneration
- Create SQLite database with proper schema
- Generate 90 days of realistic hourly energy data
- Populate database with sample records
- Run validation queries
- Preview data with pandas

## 1. Import Required Libraries

Import SQLAlchemy ORM, database utilities, datetime, pandas, and other tools needed for database initialization.

In [1]:
import os
from datetime import datetime, timedelta
import random
import numpy as np
import pandas as pd
from pathlib import Path

from sqlalchemy import create_engine, Column, Integer, String, Float, DateTime, Boolean
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, Session
from sqlalchemy.pool import StaticPool

# Create database directory if it doesn't exist
os.makedirs("data", exist_ok=True)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


## 2. Define SQLAlchemy ORM Models

Define the EnergyUsage and SolarGeneration tables with all required columns.

In [2]:
# Create declarative base for ORM models
Base = declarative_base()

class EnergyUsage(Base):
    """ORM model for energy consumption records"""
    __tablename__ = 'energy_usage'
    
    id = Column(Integer, primary_key=True)
    timestamp = Column(DateTime, nullable=False, index=True)
    device_name = Column(String, nullable=False)
    device_type = Column(String, nullable=False)
    energy_kwh = Column(Float, nullable=False)
    cost = Column(Float, nullable=False)
    peak_period = Column(Boolean, default=False)
    temperature = Column(Float)
    user_id = Column(String, default='user_001')
    
    def __repr__(self):
        return f"<EnergyUsage(timestamp={self.timestamp}, device={self.device_name}, energy={self.energy_kwh}kWh, cost=${self.cost:.2f})>"

class SolarGeneration(Base):
    """ORM model for solar generation records"""
    __tablename__ = 'solar_generation'
    
    id = Column(Integer, primary_key=True)
    timestamp = Column(DateTime, nullable=False, index=True)
    solar_kwh_generated = Column(Float, nullable=False)
    weather_condition = Column(String, nullable=False)
    battery_storage_level = Column(Float)
    exported_to_grid_kwh = Column(Float)
    
    def __repr__(self):
        return f"<SolarGeneration(timestamp={self.timestamp}, generation={self.solar_kwh_generated}kWh, weather={self.weather_condition})>"

print("✓ ORM models defined: EnergyUsage and SolarGeneration")

✓ ORM models defined: EnergyUsage and SolarGeneration


C:\Users\bhara\AppData\Local\Temp\ipykernel_18388\1091761523.py:2: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


## 3. Create SQLite Engine and Session

Initialize the SQLite database connection and configure the session factory.

In [3]:
# Create SQLite engine pointing to data/energy_data.db
db_path = "data/energy_data.db"
engine = create_engine(
    f"sqlite:///{db_path}",
    connect_args={"check_same_thread": False},
    poolclass=StaticPool,
    echo=False
)

# Create all tables defined in Base
Base.metadata.create_all(engine)

# Create session factory
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
session = SessionLocal()

print(f"✓ SQLite database created at: {db_path}")
print(f"✓ Tables created: {[table.name for table in Base.metadata.tables.values()]}")
print(f"✓ Session established")

✓ SQLite database created at: data/energy_data.db
✓ Tables created: ['energy_usage', 'solar_generation']
✓ Session established


## 4. Generate Realistic Sample Energy Usage Data

Generate 90 days of hourly energy usage data for 6 different devices with realistic consumption patterns:
- **EV Charger**: Peaks at night (18:00-23:00)
- **HVAC**: Peaks in afternoon (12:00-18:00)  
- **Dishwasher**: Evening usage (19:00-22:00)
- **Washing Machine**: Morning and evening (08:00-10:00, 19:00-21:00)
- **Pool Pump**: Afternoon operation (14:00-18:00)
- **Water Heater**: Morning, midday, and evening (07:00-09:00, 12:00-14:00, 19:00-21:00)

In [4]:
# Define device consumption patterns
devices = {
    'EV Charger': {
        'type': 'EV Charger',
        'base_kwh': 8.0,
        'variation': 2.0,
        'peak_hours': list(range(18, 24)),  # 6 PM to midnight
        'weekend_pattern': True
    },
    'HVAC': {
        'type': 'HVAC',
        'base_kwh': 2.5,
        'variation': 1.0,
        'peak_hours': list(range(12, 19)),  # Noon to 6 PM
        'weekend_pattern': False
    },
    'Dishwasher': {
        'type': 'Dishwasher',
        'base_kwh': 1.8,
        'variation': 0.3,
        'peak_hours': [19, 20, 21, 22],  # 7 PM to 10 PM
        'weekend_pattern': False
    },
    'Washing Machine': {
        'type': 'Washing Machine',
        'base_kwh': 1.5,
        'variation': 0.4,
        'peak_hours': [8, 9, 10, 19, 20, 21],  # Morning and evening
        'weekend_pattern': False
    },
    'Pool Pump': {
        'type': 'Pool Pump',
        'base_kwh': 1.0,
        'variation': 0.2,
        'peak_hours': list(range(14, 19)),  # 2 PM to 6 PM
        'weekend_pattern': True
    },
    'Water Heater': {
        'type': 'Water Heater',
        'base_kwh': 3.0,
        'variation': 0.8,
        'peak_hours': [7, 8, 9, 12, 13, 14, 19, 20, 21],  # Multiple times
        'weekend_pattern': False
    }
}

# Peak pricing hours (2 PM to 9 PM on weekdays)
peak_pricing_hours = list(range(14, 22))

# Generate 90 days of hourly data
start_date = datetime.now() - timedelta(days=90)
energy_records = []
record_count = 0

for day in range(90):
    current_date = start_date + timedelta(days=day)
    is_weekend = current_date.weekday() >= 5  # Saturday=5, Sunday=6
    
    # Ambient temperature (varies by time of day and day)
    day_temp_base = 20 + 8 * np.sin(day / 365 * 2 * np.pi)  # Seasonal variation
    
    for hour in range(24):
        timestamp = current_date.replace(hour=hour, minute=0, second=0, microsecond=0)
        
        # Temperature variation throughout the day (peak at 3 PM)
        hour_factor = np.sin((hour - 6) / 24 * 2 * np.pi)
        temperature = day_temp_base + 5 * max(0, hour_factor)
        
        # Generate usage for each device
        for device_name, device_config in devices.items():
            # Apply weekend pattern if device has one
            if device_config['weekend_pattern'] and is_weekend:
                # Higher usage on weekends
                consumption_multiplier = 1.3
            elif not device_config['weekend_pattern'] and is_weekend:
                # Lower usage on weekends for work-related devices
                consumption_multiplier = 0.6
            else:
                consumption_multiplier = 1.0
            
            # Base consumption with variation
            base = device_config['base_kwh']
            variation = random.uniform(-device_config['variation'], device_config['variation'])
            
            # Peak hour multiplier
            if hour in device_config['peak_hours']:
                hour_multiplier = random.uniform(1.2, 1.8)
            else:
                hour_multiplier = random.uniform(0.3, 0.7)
            
            # Calculate final consumption
            energy_kwh = max(0.01, (base + variation) * hour_multiplier * consumption_multiplier)
            
            # Determine if peak period and calculate cost
            is_peak = hour in peak_pricing_hours and not is_weekend
            price_per_kwh = 0.18 if is_peak else 0.12
            cost = energy_kwh * price_per_kwh
            
            # Create record
            record = EnergyUsage(
                timestamp=timestamp,
                device_name=device_name,
                device_type=device_config['type'],
                energy_kwh=round(energy_kwh, 3),
                cost=round(cost, 3),
                peak_period=is_peak,
                temperature=round(temperature, 1),
                user_id='user_001'
            )
            energy_records.append(record)
            record_count += 1

# Bulk insert all records
session.add_all(energy_records)
session.commit()

print(f"✓ Generated and inserted {record_count} energy usage records")
print(f"  Date range: {start_date.date()} to {(start_date + timedelta(days=89)).date()}")
print(f"  Devices: {', '.join(devices.keys())}")
print(f"  Records per day: {24 * len(devices)}")

✓ Generated and inserted 12960 energy usage records
  Date range: 2026-02-22 to 2026-05-22
  Devices: EV Charger, HVAC, Dishwasher, Washing Machine, Pool Pump, Water Heater
  Records per day: 144


## 5. Generate Realistic Solar Generation Data

Generate 90 days of hourly solar generation data with:
- Daytime-only generation (6 AM - 6 PM)
- Weather condition variations (sunny, partly cloudy, cloudy, rainy)
- Peak generation at noon
- Battery storage and grid export simulation

In [6]:
# Weather patterns and their impact on solar generation
weather_data = {
    'sunny': {'irradiance_mult': 1.0, 'probability': 0.35},
    'partly_cloudy': {'irradiance_mult': 0.65, 'probability': 0.35},
    'cloudy': {'irradiance_mult': 0.30, 'probability': 0.20},
    'rainy': {'irradiance_mult': 0.05, 'probability': 0.10}
}

# Solar panel specifications
max_capacity_kwh = 7.0  # Max 7 kW solar array
battery_capacity_kwh = 10.0  # 10 kWh battery

# Generate solar data for the same 90-day period
solar_records = []
generation_count = 0
battery_level = battery_capacity_kwh * 0.6  # Start at 60% charge

for day in range(90):
    current_date = start_date + timedelta(days=day)
    
    # Select daily weather based on probabilities
    weather_choice = random.choices(
        list(weather_data.keys()),
        weights=[weather_data[w]['probability'] for w in weather_data.keys()]
    )[0]
    irradiance_multiplier = weather_data[weather_choice]['irradiance_mult']
    
    # Add some day-to-day weather persistence
    if day > 0 and random.random() < 0.4 and len(solar_records) >= 24:
        weather_choice = solar_records[-24].weather_condition
        if weather_choice != 'night':
            irradiance_multiplier = weather_data[weather_choice]['irradiance_mult']
    
    for hour in range(24):
        timestamp = current_date.replace(hour=hour, minute=0, second=0, microsecond=0)
        
        # Solar generation only during daylight hours (6 AM to 6 PM)
        if 6 <= hour <= 18:
            # Solar generation curve (bell curve, peak at noon)
            hours_from_noon = abs(hour - 12)
            solar_factor = max(0, 1 - (hours_from_noon / 6.5) ** 2)
            
            # Base generation with weather and seasonal adjustments
            # Higher in summer, lower in winter
            seasonal_factor = 1 + 0.4 * np.sin((day - 80) / 365 * 2 * np.pi)
            
            generation = max_capacity_kwh * solar_factor * irradiance_multiplier * seasonal_factor
            generation = max(0, generation + random.uniform(-0.1, 0.1))  # Add noise
            
            # Battery storage logic
            available_generation = generation
            battery_charge_rate = 0.0
            exported_to_grid = 0.0
            
            # First, try to charge battery if below 80%
            if battery_level < battery_capacity_kwh * 0.8:
                charge_amount = min(available_generation * 0.9, battery_capacity_kwh - battery_level, 2.0)
                battery_charge_rate = charge_amount
                battery_level = min(battery_capacity_kwh, battery_level + charge_amount)
                available_generation -= charge_amount
            
            # Export remaining generation to grid
            if available_generation > 0:
                exported_to_grid = available_generation * 0.95  # 95% export efficiency
            
            # Update battery if discharging (night hours usage)
            # Simulated night-time consumption
            if hour >= 22 or hour <= 6:
                discharge_amount = random.uniform(0.2, 0.5)
                battery_level = max(0, battery_level - discharge_amount)
        else:
            generation = 0
            exported_to_grid = 0
            weather_choice = 'night'
            
            # Discharge battery during night
            if random.random() < 0.3:
                discharge_amount = random.uniform(0.3, 0.8)
                battery_level = max(0, battery_level - discharge_amount)
        
        # Create solar generation record
        record = SolarGeneration(
            timestamp=timestamp,
            solar_kwh_generated=round(generation, 3),
            weather_condition=weather_choice if 6 <= hour <= 18 else 'night',
            battery_storage_level=round(battery_level, 2),
            exported_to_grid_kwh=round(exported_to_grid, 3)
        )
        solar_records.append(record)
        generation_count += 1

# Bulk insert all solar records
session.add_all(solar_records)
session.commit()

print(f"✓ Generated and inserted {generation_count} solar generation records")
print(f"  Date range: {start_date.date()} to {(start_date + timedelta(days=89)).date()}")
print(f"  Weather patterns: {', '.join(weather_data.keys())}")
print(f"  Solar capacity: {max_capacity_kwh} kW")
print(f"  Battery capacity: {battery_capacity_kwh} kWh")
print(f"  Final battery level: {battery_level:.2f} kWh")

✓ Generated and inserted 2160 solar generation records
  Date range: 2026-02-22 to 2026-05-22
  Weather patterns: sunny, partly_cloudy, cloudy, rainy
  Solar capacity: 7.0 kW
  Battery capacity: 10.0 kWh
  Final battery level: 7.56 kWh


## 6. Run Validation Queries

Execute SQLAlchemy queries to validate data integrity and summarize key metrics.

In [7]:
print("=" * 70)
print("DATABASE VALIDATION QUERIES")
print("=" * 70)

# 1. Total record counts
energy_count = session.query(EnergyUsage).count()
solar_count = session.query(SolarGeneration).count()

print(f"\n1. RECORD COUNTS")
print(f"   Energy Usage Records: {energy_count:,}")
print(f"   Solar Generation Records: {solar_count:,}")
print(f"   Total Records: {energy_count + solar_count:,}")

# 2. Date range
earliest_energy = session.query(EnergyUsage).order_by(EnergyUsage.timestamp).first()
latest_energy = session.query(EnergyUsage).order_by(EnergyUsage.timestamp.desc()).first()

print(f"\n2. DATE RANGE")
print(f"   First Record: {earliest_energy.timestamp}")
print(f"   Last Record: {latest_energy.timestamp}")
print(f"   Duration: ~90 days")

# 3. Energy usage by device
print(f"\n3. ENERGY USAGE BY DEVICE")
device_stats = session.query(
    EnergyUsage.device_name,
    EnergyUsage.device_type
).distinct().all()

for device_name, device_type in sorted(device_stats):
    total_kwh = session.query(EnergyUsage).filter(
        EnergyUsage.device_name == device_name
    ).with_entities(EnergyUsage.energy_kwh).all()
    
    if total_kwh:
        total = sum([kwh[0] for kwh in total_kwh])
        cost = session.query(EnergyUsage).filter(
            EnergyUsage.device_name == device_name
        ).with_entities(EnergyUsage.cost).all()
        total_cost = sum([c[0] for c in cost])
        count = len(total_kwh)
        
        print(f"   {device_name:20s}: {total:10.2f} kWh | ${total_cost:10.2f} | {count:5d} records")

# 4. Peak vs Off-peak energy
peak_energy = session.query(EnergyUsage).filter(EnergyUsage.peak_period == True).all()
offpeak_energy = session.query(EnergyUsage).filter(EnergyUsage.peak_period == False).all()

peak_kwh = sum([e.energy_kwh for e in peak_energy])
peak_cost = sum([e.cost for e in peak_energy])
offpeak_kwh = sum([e.energy_kwh for e in offpeak_energy])
offpeak_cost = sum([e.cost for e in offpeak_energy])

print(f"\n4. PEAK VS OFF-PEAK ANALYSIS")
print(f"   Peak Period (2 PM - 9 PM):      {peak_kwh:10.2f} kWh | ${peak_cost:10.2f}")
print(f"   Off-Peak:                       {offpeak_kwh:10.2f} kWh | ${offpeak_cost:10.2f}")
print(f"   Peak Cost Ratio: {(peak_cost / (peak_cost + offpeak_cost) * 100):.1f}% of total")

# 5. Solar generation summary
solar_all = session.query(SolarGeneration).all()
total_solar = sum([s.solar_kwh_generated for s in solar_all])
total_exported = sum([s.exported_to_grid_kwh for s in solar_all])
avg_battery = np.mean([s.battery_storage_level for s in solar_all if s.battery_storage_level])

print(f"\n5. SOLAR GENERATION SUMMARY")
print(f"   Total Generated: {total_solar:.2f} kWh")
print(f"   Total Exported to Grid: {total_exported:.2f} kWh")
print(f"   Average Battery Level: {avg_battery:.2f} kWh")
print(f"   Daily Average Generation: {total_solar / 90:.2f} kWh/day")

# 6. Weather condition distribution
weather_counts = {}
for record in solar_all:
    if record.weather_condition not in weather_counts:
        weather_counts[record.weather_condition] = 0
    weather_counts[record.weather_condition] += 1

print(f"\n6. WEATHER CONDITION DISTRIBUTION")
for weather, count in sorted(weather_counts.items(), key=lambda x: x[1], reverse=True):
    if weather != 'night':
        percentage = (count / solar_count) * 100
        print(f"   {weather:15s}: {count:5d} records ({percentage:5.1f}%)")
    else:
        print(f"   {weather:15s}: {count:5d} records (night hours)")

print("\n" + "=" * 70)

DATABASE VALIDATION QUERIES

1. RECORD COUNTS
   Energy Usage Records: 12,960
   Solar Generation Records: 2,160
   Total Records: 15,120

2. DATE RANGE
   First Record: 2026-02-22 00:00:00
   Last Record: 2026-05-22 23:00:00
   Duration: ~90 days

3. ENERGY USAGE BY DEVICE
   Dishwasher          :    2301.56 kWh | $    325.51 |  2160 records
   EV Charger          :   14156.43 kWh | $   1951.64 |  2160 records
   HVAC                :    3821.35 kWh | $    546.85 |  2160 records
   Pool Pump           :    1648.78 kWh | $    232.84 |  2160 records
   Washing Machine     :    2150.91 kWh | $    298.87 |  2160 records
   Water Heater        :    5050.03 kWh | $    701.64 |  2160 records

4. PEAK VS OFF-PEAK ANALYSIS
   Peak Period (2 PM - 9 PM):         9363.80 kWh | $   1685.47
   Off-Peak:                         19765.25 kWh | $   2371.88
   Peak Cost Ratio: 41.5% of total

5. SOLAR GENERATION SUMMARY
   Total Generated: 2691.16 kWh
   Total Exported to Grid: 2387.91 kWh
   Average B

## 7. Preview Data with Pandas

Load sample data into pandas DataFrames for exploratory analysis and visualization preparation.

In [8]:
print("\n" + "=" * 70)
print("PANDAS DATA PREVIEWS")
print("=" * 70)

# 1. Load all data into pandas DataFrames
energy_data = session.query(EnergyUsage).all()
solar_data = session.query(SolarGeneration).all()

# Convert to DataFrames
energy_df = pd.DataFrame([
    {
        'timestamp': e.timestamp,
        'device_name': e.device_name,
        'device_type': e.device_type,
        'energy_kwh': e.energy_kwh,
        'cost': e.cost,
        'peak_period': e.peak_period,
        'temperature': e.temperature,
        'user_id': e.user_id
    } for e in energy_data
])

solar_df = pd.DataFrame([
    {
        'timestamp': s.timestamp,
        'solar_kwh_generated': s.solar_kwh_generated,
        'weather_condition': s.weather_condition,
        'battery_storage_level': s.battery_storage_level,
        'exported_to_grid_kwh': s.exported_to_grid_kwh
    } for s in solar_data
])

print("\n1. ENERGY USAGE TABLE (First 10 rows)")
print(energy_df.head(10).to_string())

print("\n2. ENERGY USAGE TABLE STATISTICS")
print(energy_df[['energy_kwh', 'cost', 'temperature']].describe().to_string())

print("\n\n3. SOLAR GENERATION TABLE (First 10 rows)")
print(solar_df.head(10).to_string())

print("\n4. SOLAR GENERATION TABLE STATISTICS")
print(solar_df[['solar_kwh_generated', 'battery_storage_level', 'exported_to_grid_kwh']].describe().to_string())

# Daily aggregation
print("\n\n5. DAILY ENERGY SUMMARY (Last 10 days)")
energy_df['date'] = pd.to_datetime(energy_df['timestamp']).dt.date
daily_energy = energy_df.groupby('date').agg({
    'energy_kwh': 'sum',
    'cost': 'sum',
    'temperature': 'mean'
}).tail(10)
print(daily_energy.to_string())

print("\n6. DAILY SOLAR SUMMARY (Last 10 days)")
solar_df['date'] = pd.to_datetime(solar_df['timestamp']).dt.date
daily_solar = solar_df.groupby('date').agg({
    'solar_kwh_generated': 'sum',
    'exported_to_grid_kwh': 'sum',
    'battery_storage_level': 'mean'
}).tail(10)
print(daily_solar.to_string())

# Device analysis
print("\n7. DEVICE ENERGY CONSUMPTION BREAKDOWN")
device_summary = energy_df.groupby('device_name').agg({
    'energy_kwh': ['sum', 'mean', 'count'],
    'cost': 'sum'
}).round(2)
device_summary.columns = ['Total_kWh', 'Avg_kWh', 'Records', 'Total_Cost']
print(device_summary.to_string())

# Peak period analysis
print("\n8. PEAK vs OFF-PEAK COMPARISON")
peak_summary = energy_df.groupby('peak_period').agg({
    'energy_kwh': ['sum', 'mean', 'count'],
    'cost': ['sum', 'mean']
}).round(2)
print(peak_summary.to_string())

print("\n" + "=" * 70)
print(f"✓ Database initialization complete!")
print(f"✓ Total records in database: {len(energy_df) + len(solar_df):,}")
print(f"✓ Database file: data/energy_data.db")
print("=" * 70)


PANDAS DATA PREVIEWS

1. ENERGY USAGE TABLE (First 10 rows)
            timestamp      device_name      device_type  energy_kwh   cost  peak_period  temperature   user_id
0 2026-02-22 00:00:00       EV Charger       EV Charger       6.300  0.756        False         20.0  user_001
1 2026-02-22 00:00:00             HVAC             HVAC       0.818  0.098        False         20.0  user_001
2 2026-02-22 00:00:00       Dishwasher       Dishwasher       0.429  0.051        False         20.0  user_001
3 2026-02-22 00:00:00  Washing Machine  Washing Machine       0.233  0.028        False         20.0  user_001
4 2026-02-22 00:00:00        Pool Pump        Pool Pump       0.490  0.059        False         20.0  user_001
5 2026-02-22 00:00:00     Water Heater     Water Heater       0.900  0.108        False         20.0  user_001
6 2026-02-22 01:00:00       EV Charger       EV Charger       5.159  0.619        False         20.0  user_001
7 2026-02-22 01:00:00             HVAC             